## AI Personal Finance Coach Setup

This notebook implements an AI Personal Finance Coach using **LangGraph**, a simple RAG helper, and a set of specialized agents/tools to help users manage their finances, plan ahead, and track goals.

**No paid API key required** — this version runs a small, free, open-source instruction-tuned model (`Qwen2.5-3B-Instruct`) locally on the notebook's GPU (or CPU as a fallback), instead of calling the OpenAI API. That's what was causing the `OpenAIError: Missing credentials` failure in the original version.

> Runtime: make sure you're on a GPU runtime (Runtime -> Change runtime type -> T4 GPU) for reasonable speed. It will still work on CPU, just more slowly.

In [1]:

import importlib.metadata as _metadata

_PROTECTED_PACKAGES = ["numpy", "pillow", "pandas", "scipy", "requests"]

def _get_version(pkg):
    try:
        return _metadata.version(pkg)
    except _metadata.PackageNotFoundError:
        return None

_original_versions = {p: _get_version(p) for p in _PROTECTED_PACKAGES}
print("Original (Colab-provided) versions:", _original_versions)

Original (Colab-provided) versions: {'numpy': '2.0.2', 'pillow': '11.3.0', 'pandas': '2.2.2', 'scipy': '1.16.3', 'requests': '2.32.4'}


In [2]:

!pip install -qU langgraph langchain-core transformers accelerate sentencepiece \
    pdfplumber openpyxl python-dotenv faiss-cpu pydantic \
    matplotlib reportlab tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.7/676.7 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6

In [3]:

import subprocess, sys

for _pkg, _ver in _original_versions.items():
    if _ver is None:
        continue
    current = _get_version(_pkg)
    if current != _ver:
        print(f"Restoring {_pkg}: {current} -> {_ver}")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", f"{_pkg}=={_ver}", "--force-reinstall", "--no-deps"],
            check=False,
        )
    else:
        print(f"{_pkg} unchanged ({_ver})")

numpy unchanged (2.0.2)
Restoring pillow: 12.3.0 -> 11.3.0
pandas unchanged (2.2.2)
scipy unchanged (1.16.3)
requests unchanged (2.32.4)


In [5]:
import os

MODEL_NAME_LOCAL = "Qwen/Qwen2.5-3B-Instruct"

# Paths
DATA_PATH = "data"
UPLOAD_PATH = "uploads"
VECTOR_DB = "vector_store"
REPORT_PATH = "reports"

DEFAULT_CURRENCY = "EGP"
MAX_FILE_SIZE = 20 * 1024 * 1024  # 20 MB

for p in (DATA_PATH, UPLOAD_PATH, VECTOR_DB, REPORT_PATH):
    os.makedirs(p, exist_ok=True)

### Load the Free Local LLM

We load `Qwen2.5-3B-Instruct` (Apache-2.0 licensed, no gating, no token required) once, and wrap it in a small helper (`call_local_llm`) that every agent below uses instead of `ChatOpenAI`.

We also add `extract_json_object`, a small robust parser that pulls a JSON object out of the model's reply even if it added extra text or Markdown code fences around it — local models are less reliable than GPT-4o at strict structured output, so every agent below also keeps a rule-based fallback in case parsing fails.

In [6]:
import re
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {MODEL_NAME_LOCAL} on {device}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_LOCAL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_LOCAL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
if device == "cpu":
    model.to(device)

print("Model loaded.")


def call_local_llm(system_prompt: str, user_prompt: str, max_new_tokens: int = 600, temperature: float = 0.7) -> str:
    """Calls the local free model and returns the raw generated text."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 0.01),
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def extract_json_object(text: str):
    """Robustly extracts the first valid JSON object from a text blob (handles code fences and extra prose)."""
    text = text.strip()
    text = re.sub(r"^```(?:json)?", "", text.strip())
    text = re.sub(r"```$", "", text.strip())
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                candidate = text[start:i + 1]
                try:
                    return json.loads(candidate)
                except json.JSONDecodeError:
                    break
    return None

Loading Qwen/Qwen2.5-3B-Instruct on cuda...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


### Define Financial State

The `FinanceState` TypedDict represents the complete state of our financial assistant, holding all user information, financial data, and agent outputs throughout the process.

In [7]:
from typing import TypedDict, List, Dict, Any

class FinanceState(TypedDict):
    # User Info
    age: int
    country: str
    currency: str

    # Income
    income: float

    # Expenses
    expenses: float

    # Savings
    savings: float

    # Debt
    debt: float

    # Goals
    goals: List[str]

    # Parsed Transactions
    transactions: List[Dict[str, Any]]

    # Expense Categories
    categories: Dict[str, float]

    # Financial Score
    financial_score: int
    assessment_summary: str

    # CashFlow
    cashflow: float

    # Leakage Detection
    leakages: List[str]

    # RAG Context
    rag_context: str

    # Investment Plan
    investment_plan: Dict[str, Any]

    # Simulation Results
    simulations: List[Dict]

    # Final Report
    report: Dict[str, Any]

### Financial Analyzer Agent

Analyzes the current financial state and produces a financial score (0-100) and a short assessment, using the local LLM. Falls back to a safe default if the model output can't be parsed.

In [8]:
def financial_analyzer(state: FinanceState) -> FinanceState:
    """
    Analyzes the current financial state and performs an initial assessment using the local LLM.
    Provides a financial score and an initial overview.
    """
    print("Executing Financial Analyzer Agent...")

    age = state.get("age", "N/A")
    income = state.get("income", 0.0)
    expenses = state.get("expenses", 0.0)
    goals = state.get("goals", ["no specific goals provided"])
    currency = state.get("currency", DEFAULT_CURRENCY)

    system_prompt = (
        "You are an AI financial expert. Analyze the user's financial situation and provide a "
        "financial score (out of 100) and a short assessment. Respond ONLY with a valid JSON "
        "object with exactly these keys: 'financial_score' (int) and 'assessment_summary' (str). "
        "No extra text, no markdown."
    )
    user_prompt = (
        f"Here is my financial data:\nAge: {age}\nIncome: {income} {currency}\n"
        f"Expenses: {expenses} {currency}\nGoals: {', '.join(goals)}\n\n"
        "Provide a financial score and an assessment summary in JSON format."
    )

    try:
        raw_response = call_local_llm(system_prompt, user_prompt, max_new_tokens=250)
        response = extract_json_object(raw_response)
        if response is None:
            raise ValueError(f"Could not parse JSON from model output: {raw_response!r}")
        print(f"LLM Response: {response}")

        state["financial_score"] = int(response.get("financial_score", state.get("financial_score", 60)))
        state["assessment_summary"] = response.get("assessment_summary", "Initial assessment not available.")

    except Exception as e:
        print(f"Error during LLM invocation for financial analyzer: {e}")
        state["financial_score"] = state.get("financial_score", 60)
        state["assessment_summary"] = "Failed to get LLM assessment, using fallback values."

    if state.get("income") is not None and state.get("expenses") is not None:
        state["cashflow"] = state["income"] - state["expenses"]
    else:
        state["cashflow"] = 0.0

    print(f"Financial Analyzer updated state: {state}")
    return state

### Expense Categorizer Agent

Classifies financial transactions into standard categories (Housing, Food, Transportation, etc.) using the local LLM. Uses a small set of dummy transactions when none are supplied.

In [9]:
def expense_categorizer(state: FinanceState) -> FinanceState:
    """
    Categorizes transactions into expense categories using the local LLM.
    """
    print("Executing Expense Categorizer Agent...")

    transactions_data = state.get("transactions") or [
        {"item": "Groceries from Carrefour", "amount": 300.0},
        {"item": "Monthly Rent", "amount": 1000.0},
        {"item": "Uber Ride", "amount": 50.0},
        {"item": "Electricity Bill", "amount": 100.0},
        {"item": "Netflix Subscription", "amount": 20.0},
        {"item": "Dinner at Restaurant", "amount": 100.0},
        {"item": "New Gadget Online Store", "amount": 250.0},
    ]

    transactions_str = "\n".join(
        [f"- {t['item']}: {t['amount']} {state.get('currency', DEFAULT_CURRENCY)}" for t in transactions_data]
    )

    system_prompt = (
        "You are an AI financial assistant. Categorize the given transactions into standard "
        "financial categories like 'Housing', 'Food', 'Transportation', 'Utilities', "
        "'Entertainment', 'Shopping', 'Debt Payments', 'Savings', 'Other'. Respond ONLY with a "
        "valid JSON object where keys are categories and values are the summed numeric amounts "
        "for that category. If a transaction doesn't fit well, put it in 'Other'. No extra text."
    )
    user_prompt = f"Categorize the following transactions:\n{transactions_str}"

    try:
        raw_response = call_local_llm(system_prompt, user_prompt, max_new_tokens=300, temperature=0.3)
        categorized_expenses = extract_json_object(raw_response)
        if categorized_expenses is None:
            raise ValueError(f"Could not parse JSON from model output: {raw_response!r}")
        print(f"LLM Categorization: {categorized_expenses}")

        if "categories" not in state or not state["categories"]:
            state["categories"] = {}
        for category, amount in categorized_expenses.items():
            state["categories"][category] = state["categories"].get(category, 0.0) + float(amount)

    except Exception as e:
        print(f"Error during LLM invocation for expense categorizer: {e}")
        state["categories"] = {
            "Food": 300.0,
            "Housing": 1000.0,
            "Transportation": 150.0,
            "Utilities": 100.0,
            "Entertainment": 120.0,
            "Shopping": 250.0,
            "Other": 50.0,
        }
        print("Falling back to dummy categorization due to LLM error.")

    print(f"Expense Categorizer updated state with categories: {state['categories']}")
    return state

### Leakage Detector Agent

Analyzes categorized expenses to identify potential financial leakages (excessive/unnecessary spending), using the local LLM.

In [10]:
def leakage_detector(state: FinanceState) -> FinanceState:
    """
    Identifies financial leakages based on categorized expenses using the local LLM.
    """
    print("Executing Leakage Detector Agent...")

    categories = state.get("categories", {})
    total_expenses = sum(categories.values())

    if not categories:
        state["leakages"] = ["No categorized expenses available to detect leakages."]
        print("No categories found for leakage detection.")
        return state

    categories_str = "\n".join(
        [f"- {cat}: {amount:.2f} {state.get('currency', DEFAULT_CURRENCY)}" for cat, amount in categories.items()]
    )

    system_prompt = (
        "You are an AI financial expert specializing in identifying financial leakages. Analyze "
        "the user's categorized expenses and identify areas where spending might be excessive, "
        "unnecessary, or could be reduced. Respond ONLY with a valid JSON object with a single "
        "key 'leakages' which is a list of short, concise, actionable strings. No extra text."
    )
    user_prompt = (
        f"Here are my categorized monthly expenses:\n{categories_str}\n\n"
        f"Total monthly expenses: {total_expenses:.2f} {state.get('currency', DEFAULT_CURRENCY)}\n\n"
        "Identify potential financial leakages and provide actionable advice to reduce them."
    )

    try:
        raw_response = call_local_llm(system_prompt, user_prompt, max_new_tokens=300, temperature=0.5)
        response = extract_json_object(raw_response)
        if response is None:
            raise ValueError(f"Could not parse JSON from model output: {raw_response!r}")
        state["leakages"] = response.get("leakages", []) or ["No significant leakages detected based on current expenses."]

    except Exception as e:
        print(f"Error during LLM invocation for leakage detector: {e}")
        leakages_found_fallback = []
        if categories.get("Entertainment", 0) > 150.0 and total_expenses > 0:
            leakages_found_fallback.append("High spending on Entertainment (e.g., streaming services, dining out).")
        if categories.get("Other", 0) > 75.0 and total_expenses > 0:
            leakages_found_fallback.append("Uncategorized 'Other' expenses are significant, suggesting potential hidden spending.")
        if not leakages_found_fallback:
            leakages_found_fallback.append("No significant leakages detected with fallback logic.")
        state["leakages"] = leakages_found_fallback
        print("Falling back to dummy leakage detection due to LLM error.")

    print(f"Leakage Detector updated state with leakages: {state['leakages']}")
    return state

### Simple RAG Retrieval Function (Placeholder)

Simulates querying a knowledge base with a given query and returning relevant financial guidance. In a full implementation, this would embed the query and perform a similarity search over a real vector database (FAISS, Chroma, Pinecone, etc.).

In [11]:
def rag_retriever(query: str) -> str:
    """
    Simulates a RAG retrieval process by returning canned context based on keywords in the query.
    In a real implementation, this would query a vector database.
    """
    print(f"Executing RAG Retriever with query: '{query}'...")
    q = query.lower()
    if "budgeting" in q or "expenses" in q:
        return ("Budgeting tips: Categorize all expenses, set spending limits, and review regularly. "
                 "Prioritize needs over wants. Use the 50/30/20 rule (50% needs, 30% wants, 20% savings/debt).")
    elif "investment" in q or "stocks" in q or "funds" in q:
        return ("Investment advice: Diversify your portfolio, understand your risk tolerance, and consider "
                 "long-term goals. Explore mutual funds, ETFs, and local market opportunities.")
    elif "goals" in q or "planning" in q:
        return ("Financial goal planning: Define SMART goals (Specific, Measurable, Achievable, Relevant, "
                 "Time-bound). Break down large goals into smaller, manageable steps. Automate savings towards goals.")
    else:
        return "General financial advice: Maintain an emergency fund, pay off high-interest debt, and regularly review your financial health."

### RAG Context Retriever Agent

Formulates a query based on the current state (leakages, goals, financial score) and fetches relevant context via `rag_retriever` above.

In [12]:
def retrieve_rag_context(state: FinanceState) -> FinanceState:
    """
    Retrieves relevant financial information (RAG context) based on the current state's needs.
    """
    print("Executing RAG Context Retrieval Agent...")

    query_parts = []
    if state.get("leakages"):
        query_parts.append("budgeting strategies for reducing expenses")
    if state.get("goals"):
        query_parts.append("investment options and savings strategies for goals")
    if state.get("financial_score") is not None and state["financial_score"] < 70:
        query_parts.append("improving financial health")

    if query_parts:
        combined_query = ". ".join(query_parts)
        state["rag_context"] = rag_retriever(combined_query)
    else:
        state["rag_context"] = rag_retriever("general financial advice")

    print(f"RAG Context retrieved: {state['rag_context']}")
    return state

### Cash Flow and Forecasting Tool

Calculates current cash flow and projects a simple month-by-month forecast. This is plain arithmetic (no LLM needed) and is used by the Financial Planner agent below.

In [13]:
def calculate_and_forecast_cashflow(state: FinanceState, forecast_months: int = 12) -> Dict[str, Any]:
    """
    Calculates current cash flow and provides a basic forecast.
    """
    print(f"Executing Cash Flow Forecaster Tool for {forecast_months} months...")

    current_income = state.get("income", 0.0)
    current_expenses = sum(state.get("categories", {}).values())
    if current_expenses == 0.0:
        current_expenses = state.get("expenses", 0.0)

    monthly_cashflow = current_income - current_expenses
    projected_savings = state.get("savings", 0.0)

    forecast = {
        "current_monthly_cashflow": monthly_cashflow,
        "forecast_period_months": forecast_months,
        "projected_cashflow": {},
        "projected_savings_growth": {},
    }

    for month in range(1, forecast_months + 1):
        projected_savings += monthly_cashflow
        forecast["projected_cashflow"][f"Month {month}"] = monthly_cashflow
        forecast["projected_savings_growth"][f"Month {month}"] = projected_savings

    print(f"Cash Flow Forecast: {forecast}")
    return forecast

### Financial Planning Agent

Uses the financial analysis, categorized expenses, leakages, RAG context, and cash flow forecast to create a personalized financial plan (savings target, investment recommendation, goal timeline, tips) via the local LLM.

In [14]:
def financial_planner(state: FinanceState) -> FinanceState:
    """
    Generates a customized financial plan for the user using the local LLM.
    """
    print("Executing Financial Planner Agent...")

    cashflow_forecast = calculate_and_forecast_cashflow(state, forecast_months=24)

    financial_score = state.get("financial_score", "N/A")
    categories = state.get("categories", {})
    leakages = state.get("leakages", [])
    goals = state.get("goals", ["no specific goals provided"])
    rag_context = state.get("rag_context", "No specific financial advice context available.")
    currency = state.get("currency", DEFAULT_CURRENCY)

    categories_str = "\n".join([f"- {cat}: {amount:.2f} {currency}" for cat, amount in categories.items()])
    leakages_str = "; ".join(leakages) if leakages else "None identified."
    goals_str = ", ".join(goals)

    system_prompt = (
        "You are an AI financial planner. Create a comprehensive financial plan based on the "
        "user's data. Respond ONLY with a valid JSON object with keys: 'summary' (str), "
        "'monthly_saving_target' (float), 'investment_strategy_recommendation' (str), "
        "'goal_timeline_estimate' (an object mapping each goal to a short str estimate), "
        "'immediate_tips' (a list of str). No extra text."
    )
    user_prompt = (
        f"Financial Score: {financial_score}/100\nCategorized Expenses:\n{categories_str}\n"
        f"Identified Leakages: {leakages_str}\nUser Goals: {goals_str}\nRAG Context: {rag_context}\n"
        f"Monthly Cashflow: {cashflow_forecast['current_monthly_cashflow']:.2f} {currency}\n\n"
        "Create a detailed financial plan and return it as JSON."
    )

    try:
        raw_response = call_local_llm(system_prompt, user_prompt, max_new_tokens=500)
        plan_details = extract_json_object(raw_response)
        if plan_details is None:
            raise ValueError(f"Could not parse JSON from model output: {raw_response!r}")
        print(f"LLM Financial Plan: {plan_details}")

        plan_details["monthly_saving_target"] = float(plan_details.get("monthly_saving_target", 0.0))
        state["investment_plan"] = plan_details

    except Exception as e:
        print(f"Error during LLM invocation for financial planner: {e}")
        plan_details_fallback = {
            "summary": "Failed to generate LLM plan, providing a basic fallback.",
            "monthly_saving_target": max(0.0, cashflow_forecast["current_monthly_cashflow"] * 0.5),
            "investment_strategy_recommendation": "Review basics, emergency fund first.",
            "goal_timeline_estimate": {"General savings": "Ongoing"},
            "immediate_tips": ["Re-evaluate unnecessary subscriptions."],
        }
        if leakages:
            plan_details_fallback["immediate_tips"].append(f"Address detected leakages like: {leakages_str}.")
        state["investment_plan"] = plan_details_fallback
        print("Falling back to dummy financial plan due to LLM error.")

    print(f"Financial Planner updated state with plan: {state['investment_plan']}")
    return state

### Investment Comparison Tool

A lightweight lookup tool comparing investment options by risk tolerance. Used on demand (not part of the main linear graph).

In [15]:
def compare_investment_options(state: FinanceState, risk_tolerance: str = "moderate") -> Dict[str, Any]:
    """
    Compares investment options based on risk tolerance.
    """
    print(f"Executing Investment Comparison Tool for risk tolerance: {risk_tolerance}...")

    investment_options = {
        "conservative": {"name": "Government Bonds & Fixed Deposits", "expected_return_annual": 0.05, "risk_level": "Low"},
        "moderate": {"name": "Diversified Mutual Funds & Real Estate", "expected_return_annual": 0.08, "risk_level": "Medium"},
        "aggressive": {"name": "Stocks & Venture Capital", "expected_return_annual": 0.12, "risk_level": "High"},
    }

    selected_option = investment_options.get(risk_tolerance.lower(), investment_options["moderate"])
    print(f"Investment Comparison: {selected_option}")
    return selected_option

### Scenario Simulation Agent

Simulates a financial scenario (salary increase, inflation, job loss, etc.) and projects its impact using the local LLM. Because it needs an extra `scenario` argument, we wrap it in `simulation_node` (below) with a default baseline scenario so it can plug into the LangGraph pipeline like every other node.

In [16]:
def simulation_agent(state: FinanceState, scenario: Dict[str, Any]) -> FinanceState:
    """
    Simulates a given financial scenario and updates the state with results using the local LLM.
    """
    print(f"Executing Simulation Agent for scenario: {scenario.get('name', 'Custom Scenario')}...")

    initial_income = state.get("income", 0.0)
    initial_expenses = sum(state.get("categories", {}).values()) or state.get("expenses", 0.0)
    initial_savings = state.get("savings", 0.0)
    initial_cashflow = initial_income - initial_expenses
    current_goals = state.get("goals", ["no specific goals provided"])
    currency = state.get("currency", DEFAULT_CURRENCY)

    scenario_description = scenario.get("description", "A custom financial scenario.")

    system_prompt = (
        "You are an AI financial simulation expert. Analyze the user's current financial state "
        "and a given scenario. Project the impact of this scenario on their finances. Respond "
        "ONLY with a valid JSON object with keys: 'scenario_name' (str), 'simulated_income' (float), "
        "'simulated_expenses' (float), 'simulated_cashflow' (float), "
        "'projected_savings_after_scenario' (float), 'impact_summary' (str), "
        "'recommendations_for_scenario' (list of str). No extra text."
    )
    user_prompt = (
        f"Current Financial State:\nIncome: {initial_income} {currency}\nExpenses: {initial_expenses} {currency}\n"
        f"Cashflow: {initial_cashflow} {currency}\nSavings: {initial_savings} {currency}\n"
        f"Goals: {', '.join(current_goals)}\n\nScenario to simulate: {scenario_description}\n\n"
        "Project the financial impact and provide recommendations in JSON format."
    )

    try:
        raw_response = call_local_llm(system_prompt, user_prompt, max_new_tokens=400)
        simulation_result = extract_json_object(raw_response)
        if simulation_result is None:
            raise ValueError(f"Could not parse JSON from model output: {raw_response!r}")
        print(f"LLM Simulation Result: {simulation_result}")

        simulation_result["simulated_income"] = float(simulation_result.get("simulated_income", initial_income))
        simulation_result["simulated_expenses"] = float(simulation_result.get("simulated_expenses", initial_expenses))
        simulation_result["simulated_cashflow"] = float(simulation_result.get("simulated_cashflow", initial_cashflow))
        simulation_result["projected_savings_after_scenario"] = float(
            simulation_result.get("projected_savings_after_scenario", initial_savings)
        )

    except Exception as e:
        print(f"Error during LLM invocation for simulation agent: {e}")
        simulated_income = initial_income * (1 + scenario.get("income_change_percent", 0))
        simulated_expenses = initial_expenses * (1 + scenario.get("expenses_change_percent", 0))
        simulated_cashflow = simulated_income - simulated_expenses
        simulated_savings = initial_savings + (simulated_cashflow * scenario.get("months_to_simulate", 1))

        simulation_result = {
            "scenario_name": scenario.get("name", "Custom Scenario") + " (Fallback)",
            "simulated_income": simulated_income,
            "simulated_expenses": simulated_expenses,
            "simulated_cashflow": simulated_cashflow,
            "projected_savings_after_scenario": simulated_savings,
            "impact_summary": "Failed to get LLM simulation, using basic calculation.",
            "recommendations_for_scenario": ["Consider the basic changes in income/expenses provided."],
        }
        print("Falling back to dummy simulation due to LLM error.")

    state.setdefault("simulations", []).append(simulation_result)
    print(f"Simulation Agent updated state with results: {simulation_result['scenario_name']}")
    return state


def simulation_node(state: FinanceState) -> FinanceState:
    """
    Graph-compatible wrapper around simulation_agent: LangGraph nodes only receive `state`,
    so this supplies a default baseline scenario. Call simulation_agent directly with your
    own scenario dict for custom what-if analysis (see the example near the end of the notebook).
    """
    default_scenario = {
        "name": "Default Scenario Check",
        "description": "No specific scenario provided; performing a baseline check of current finances.",
        "income_change_percent": 0.0,
        "expenses_change_percent": 0.0,
        "months_to_simulate": 1,
    }
    return simulation_agent(state, default_scenario)

### Output Parser / Report Generator Agent

Compiles everything gathered by the previous agents into a comprehensive report using the local LLM, and exports it as a PDF via `reportlab`.

In [17]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
import os


def report_generator(state: FinanceState) -> FinanceState:
    """
    Generates a comprehensive financial report based on the final state using the local LLM,
    and exports it as a PDF.
    """
    print("Executing Report Generator Agent...")

    state_summary = {
        "user_info": {
            "age": state.get("age", "N/A"),
            "country": state.get("country", "N/A"),
            "currency": state.get("currency", DEFAULT_CURRENCY),
        },
        "income": state.get("income", 0.0),
        "expenses": state.get("expenses", 0.0),
        "savings": state.get("savings", 0.0),
        "debt": state.get("debt", 0.0),
        "goals": state.get("goals", ["Not specified"]),
        "categorized_expenses": state.get("categories", {}),
        "financial_score": state.get("financial_score", "N/A"),
        "cashflow": state.get("cashflow", 0.0),
        "leakages": state.get("leakages", ["None identified"]),
        "rag_context": state.get("rag_context", "No specific financial advice context available."),
        "investment_plan": state.get("investment_plan", {"summary": "No plan generated yet"}),
        "simulations": state.get("simulations", []),
    }

    system_prompt = (
        "You are an AI financial coach and report generator. Compile all provided financial "
        "data, analysis, and plans into a comprehensive, actionable financial report. Respond "
        "ONLY with a valid JSON object with keys: 'overall_summary', "
        "'financial_health_score_details', 'expense_analysis', 'leakage_insights', "
        "'saving_plan_details', 'investment_strategy', 'goal_progress_notes', "
        "'simulation_results_summary', 'actionable_recommendations' (list of str), "
        "'important_notes' (list of str). Keep each value concise. No extra text."
    )
    user_prompt = f"Here is the complete financial state of the user: {json.dumps(state_summary)}\n\nGenerate a detailed financial report in JSON format."

    try:
        raw_response = call_local_llm(system_prompt, user_prompt, max_new_tokens=800)
        report_content = extract_json_object(raw_response)
        if report_content is None:
            raise ValueError(f"Could not parse JSON from model output: {raw_response!r}")
        print(f"LLM Generated Report Content: {report_content}")

    except Exception as e:
        print(f"Error during LLM invocation for report generator: {e}")
        report_content = {
            "overall_summary": (
                f"Failed to generate detailed LLM report. Basic overview: Financial health score is "
                f"{state.get('financial_score', 'N/A')}/100. Monthly cashflow is "
                f"{state.get('cashflow', 0.0):.2f} {state.get('currency', DEFAULT_CURRENCY)}."
            ),
            "financial_health_score_details": "Details unavailable due to LLM error.",
            "expense_analysis": "Categorized expenses not analyzed by LLM.",
            "leakage_insights": state.get("leakages", ["No significant leakages detected."]),
            "saving_plan_details": state.get("investment_plan", {}).get("monthly_saving_target", "N/A"),
            "investment_strategy": state.get("investment_plan", {}).get("investment_strategy_recommendation", "N/A"),
            "goal_progress_notes": state.get("investment_plan", {}).get("goal_timeline_estimate", "N/A"),
            "simulation_results_summary": "Simulations not summarized by LLM.",
            "actionable_recommendations": state.get("investment_plan", {}).get("immediate_tips", [])
            + ["Re-evaluate unnecessary subscriptions and try again."],
            "important_notes": ["Basic report generated due to an issue with AI analysis. Please provide more specific data if possible."],
        }
        print("Falling back to dummy report content due to LLM error.")

    if not os.path.exists(REPORT_PATH):
        os.makedirs(REPORT_PATH)
    pdf_filename = os.path.join(REPORT_PATH, "Financial_Report.pdf")

    doc = SimpleDocTemplate(pdf_filename, pagesize=letter)
    styles = getSampleStyleSheet()
    story = []

    def add_section(title, body):
        story.append(Paragraph(title, styles["h2"]))
        story.append(Paragraph(str(body), styles["Normal"]))
        story.append(Spacer(1, 8))

    story.append(Paragraph("AI Personal Finance Coach - Comprehensive Financial Report", styles["h1"]))
    story.append(Spacer(1, 12))

    add_section("1. Overall Summary", report_content.get("overall_summary", "No overall summary provided."))

    story.append(Paragraph("2. Financial Health Score & Details", styles["h2"]))
    story.append(Paragraph(f"Score: {state.get('financial_score', 'N/A')}/100", styles["h3"]))
    story.append(Paragraph(str(report_content.get("financial_health_score_details", "Details unavailable.")), styles["Normal"]))
    story.append(Spacer(1, 8))

    story.append(Paragraph("3. Expense Analysis", styles["h2"]))
    story.append(Paragraph(str(report_content.get("expense_analysis", "Details unavailable.")), styles["Normal"]))
    if state.get("categories"):
        story.append(Paragraph("Categorized Expenses:", styles["h3"]))
        for cat, amount in state["categories"].items():
            story.append(Paragraph(f"- {cat}: {amount:.2f} {state.get('currency', DEFAULT_CURRENCY)}", styles["Normal"]))
    story.append(Spacer(1, 8))

    add_section("4. Leakage Insights", report_content.get("leakage_insights", "No leakage insights provided."))
    add_section("5. Saving Plan Details", report_content.get("saving_plan_details", "No saving plan details provided."))
    add_section("6. Investment Strategy & Recommendations", report_content.get("investment_strategy", "No investment strategy provided."))
    add_section("7. Goal Progress Notes", report_content.get("goal_progress_notes", "No goal progress notes provided."))
    add_section("8. Simulation Results Summary", report_content.get("simulation_results_summary", "No simulation results summary provided."))

    story.append(Paragraph("9. Actionable Recommendations", styles["h2"]))
    for rec in report_content.get("actionable_recommendations", []):
        story.append(Paragraph(f"- {rec}", styles["Normal"]))
    story.append(Spacer(1, 8))

    story.append(Paragraph("10. Important Notes", styles["h2"]))
    for note in report_content.get("important_notes", []):
        story.append(Paragraph(f"- {note}", styles["Normal"]))
    story.append(Spacer(1, 12))

    doc.build(story)

    report_content["pdf_report_path"] = pdf_filename
    state["report"] = report_content

    print(f"Report Generator updated state with final report and generated PDF: {pdf_filename}")
    return state

### Build the LangGraph StateGraph

Now that every agent is defined, we build the graph **once**, in the correct order:

`Financial Analyzer -> Expense Categorizer -> Leakage Detector -> RAG Context Retriever -> Financial Planner -> Simulation Agent -> Report Generator -> END`

In [18]:
from langgraph.graph import StateGraph, END

builder = StateGraph(FinanceState)

builder.add_node("Financial Analyzer", financial_analyzer)
builder.add_node("Expense Categorizer", expense_categorizer)
builder.add_node("Leakage Detector", leakage_detector)
builder.add_node("RAG Context Retriever", retrieve_rag_context)
builder.add_node("Financial Planner", financial_planner)
builder.add_node("Simulation Agent", simulation_node)
builder.add_node("Report Generator", report_generator)

builder.set_entry_point("Financial Analyzer")

builder.add_edge("Financial Analyzer", "Expense Categorizer")
builder.add_edge("Expense Categorizer", "Leakage Detector")
builder.add_edge("Leakage Detector", "RAG Context Retriever")
builder.add_edge("RAG Context Retriever", "Financial Planner")
builder.add_edge("Financial Planner", "Simulation Agent")
builder.add_edge("Simulation Agent", "Report Generator")
builder.add_edge("Report Generator", END)

graph = builder.compile()

print("LangGraph StateGraph initialized and compiled with the full workflow.")
print("Workflow: Financial Analyzer -> Expense Categorizer -> Leakage Detector -> "
      "RAG Context Retriever -> Financial Planner -> Simulation Agent -> Report Generator -> END")

LangGraph StateGraph initialized and compiled with the full workflow.
Workflow: Financial Analyzer -> Expense Categorizer -> Leakage Detector -> RAG Context Retriever -> Financial Planner -> Simulation Agent -> Report Generator -> END


### Test the Financial Coach Graph

Now that all agents and the graph are set up, let's run a test with some sample user data.

In [19]:
test_state = FinanceState(
    age=30,
    country="Egypt",
    currency=DEFAULT_CURRENCY,
    income=5000.0,
    expenses=2500.0,
    savings=10000.0,
    debt=5000.0,
    goals=["Buy a house", "Save for retirement"],
    transactions=[],
    categories={},
    financial_score=0,
    assessment_summary="",
    cashflow=0.0,
    leakages=[],
    rag_context="",
    investment_plan={},
    simulations=[],
    report={},
)

print("Initial Test State:")
print(test_state)

Initial Test State:
{'age': 30, 'country': 'Egypt', 'currency': 'EGP', 'income': 5000.0, 'expenses': 2500.0, 'savings': 10000.0, 'debt': 5000.0, 'goals': ['Buy a house', 'Save for retirement'], 'transactions': [], 'categories': {}, 'financial_score': 0, 'assessment_summary': '', 'cashflow': 0.0, 'leakages': [], 'rag_context': '', 'investment_plan': {}, 'simulations': [], 'report': {}}


In [20]:
final_state = graph.invoke(test_state)

print("\nFinal State after graph execution:")
print(final_state)

Executing Financial Analyzer Agent...
LLM Response: {'financial_score': 45, 'assessment_summary': 'The user has a moderate financial score given their current income and expenses. They are on track to meet their goals but have room for improvement in managing finances more efficiently.'}
Financial Analyzer updated state: {'age': 30, 'country': 'Egypt', 'currency': 'EGP', 'income': 5000.0, 'expenses': 2500.0, 'savings': 10000.0, 'debt': 5000.0, 'goals': ['Buy a house', 'Save for retirement'], 'transactions': [], 'categories': {}, 'financial_score': 45, 'assessment_summary': 'The user has a moderate financial score given their current income and expenses. They are on track to meet their goals but have room for improvement in managing finances more efficiently.', 'cashflow': 2500.0, 'leakages': [], 'rag_context': '', 'investment_plan': {}, 'simulations': [], 'report': {}}
Executing Expense Categorizer Agent...
LLM Categorization: {'Housing': 1000.0, 'Food': 300.0, 'Transportation': 50.0, 

### Summary of the Generated Report

Below is a summary of the key findings from the financial report generated. The full report is also available as a PDF at the path printed below.

In [21]:
if final_state.get("report"):
    print("Overall Summary:", final_state["report"].get("overall_summary"))
    print("Financial Health Score:", final_state["report"].get("financial_health_score_details"))
    print("Expense Analysis Summary:", final_state["report"].get("expense_analysis"))
    print("Leakage Insights:", final_state["report"].get("leakage_insights"))
    print("Saving Plan Details:", final_state["report"].get("saving_plan_details"))
    print("Investment Strategy:", final_state["report"].get("investment_strategy"))
    print("Actionable Recommendations:", final_state["report"].get("actionable_recommendations"))
    print(f"PDF Report Path: {final_state['report'].get('pdf_report_path')}")
else:
    print("No report generated in the final state.")

Overall Summary: The user's financial score is 45 out of 100, indicating room for improvement in managing expenses and increasing savings. The cashflow is positive at 2500 EGP per month, with a focus on reducing shopping and entertainment expenses.
Financial Health Score: The financial health score is moderate, suggesting that the user should focus on better managing expenses, particularly on high spending categories like shopping and entertainment. The user's immediate goal is to save more by cutting down these expenses and increasing their monthly saving target to 318 EGP.
Expense Analysis Summary: {'total_expenses': 2500.0, 'categorized_expenses': {'Housing': 1000.0, 'Food': 300.0, 'Transportation': 50.0, 'Utilities': 100.0, 'Entertainment': 120.0, 'Shopping': 250.0, 'Other': 0.0}}
Leakage Insights: ['High shopping expenses (Shopping: 250.00 EGP)', 'Consider reducing entertainment budget (Entertainment: 120.00 EGP)']
Saving Plan Details: {'current_savings': 10000.0, 'monthly_saving_

### Bonus: Running a Custom What-If Scenario

`simulation_agent` (used inside the graph via `simulation_node`) can also be called directly with your own scenario — e.g. a salary cut or a big expense — to see the projected impact without re-running the whole graph.

In [22]:
custom_scenario = {
    "name": "Job Loss - 3 Months",
    "description": "User loses their job and has no income for 3 months, relying on savings.",
    "income_change_percent": -1.0,
    "expenses_change_percent": -0.1,
    "months_to_simulate": 3,
}

final_state = simulation_agent(final_state, custom_scenario)
print("\nLatest simulation:", final_state["simulations"][-1])

Executing Simulation Agent for scenario: Job Loss - 3 Months...
LLM Simulation Result: {'scenario_name': 'Losing Job for 3 Months', 'simulated_income': 0.0, 'simulated_expenses': 1820.0, 'simulated_cashflow': -1820.0, 'projected_savings_after_scenario': 8200.0, 'impact_summary': 'The user will deplete their savings by 20% over the course of 3 months, leaving them with only 8200 EGP.', 'recommendations_for_scenario': ['Prioritize essential expenses.', 'Review and reduce non-essential spending.', 'Consider temporary side jobs or freelance work.', 'Explore short-term financial assistance options.']}
Simulation Agent updated state with results: Losing Job for 3 Months

Latest simulation: {'scenario_name': 'Losing Job for 3 Months', 'simulated_income': 0.0, 'simulated_expenses': 1820.0, 'simulated_cashflow': -1820.0, 'projected_savings_after_scenario': 8200.0, 'impact_summary': 'The user will deplete their savings by 20% over the course of 3 months, leaving them with only 8200 EGP.', 'recom